In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

In [ ]:
SEED = 42

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 0.0001

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

In [ ]:
DATASET_PATH = Path("../Chest_XRay_Pneumonia")
TRAIN_PATH = DATASET_PATH / "train"
TEST_PATH = DATASET_PATH / "test"

MODEL_PATH = "pneumonia_resnet18.pth"
PREDICTIONS_PLOT_PATH = "pneumonia_predictions.png"

In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
train_dataset = datasets.ImageFolder(
    root=TRAIN_PATH,
    transform=transform
)

test_dataset = datasets.ImageFolder(
    root=TEST_PATH,
    transform=transform
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
print("Train images:", len(train_dataset))
print("Test images:", len(test_dataset))

print(train_dataset.class_to_idx)

In [ ]:

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)


for param in model.parameters():
    param.requires_grad = False


model.fc = nn.Linear(512, 2)


model = model.to(DEVICE)


criterion = nn.CrossEntropyLoss()


optimizer = optim.Adam(
    model.fc.parameters(),
    lr=LEARNING_RATE
)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()

        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [ ]:
def evaluate_loss_accuracy(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()

            total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [ ]:
history = {
    "train_loss": [],
    "train_acc": [],
    "test_loss": [],
    "test_acc": [],
}

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        DEVICE,
    )

    test_loss, test_acc = evaluate_loss_accuracy(
        model,
        test_loader,
        criterion,
        DEVICE,
    )

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss={train_loss:.4f} | "
        f"Train Acc={train_acc:.4f} | "
        f"Test Loss={test_loss:.4f} | "
        f"Test Acc={test_acc:.4f}"
    )

history_df = pd.DataFrame(history)

display(history_df)

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history_df["train_loss"], label="Train")
plt.plot(history_df["test_loss"], label="Test")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history_df["train_acc"], label="Train")
plt.plot(history_df["test_acc"], label="Test")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
def collect_predictions(model, loader, device):

    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            outputs = model(images)

            predictions = outputs.argmax(dim=1).cpu().numpy()

            y_true.extend(labels.numpy().tolist())
            y_pred.extend(predictions.tolist())

    return np.array(y_true), np.array(y_pred)

In [ ]:
y_true, y_pred = collect_predictions(
    model,
    test_loader,
    DEVICE,
)

positive_label = train_dataset.class_to_idx["PNEUMONIA"]

metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(
        y_true,
        y_pred,
        pos_label=positive_label,
        zero_division=0,
    ),
    "recall": recall_score(
        y_true,
        y_pred,
        pos_label=positive_label,
        zero_division=0,
    ),
    "f1_score": f1_score(
        y_true,
        y_pred,
        pos_label=positive_label,
        zero_division=0,
    ),
}

pd.DataFrame([metrics]).round(4)

In [ ]:
class_names = list(train_dataset.class_to_idx.keys())

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        zero_division=0,
    )
)

In [ ]:
cm = confusion_matrix(y_true, y_pred)

display_cm = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names,
)

display_cm.plot(
    cmap="Blues",
    values_format="d",
)

plt.title("Confusion Matrix")
plt.show()

In [ ]:
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "class_to_idx": train_dataset.class_to_idx,
        "image_size": IMG_SIZE,
        "model_name": "resnet18",
    },
    MODEL_PATH,
)

print(f"Model saved to {MODEL_PATH}")

In [ ]:
def denormalize_image(image):

    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std = torch.tensor([0.229,0.224,0.225]).view(3,1,1)

    image = image.cpu() * std + mean

    return torch.clamp(image,0,1)

In [ ]:
def show_random_predictions(
    model,
    dataset,
    class_names,
    device,
    n_images=8,
):

    model.eval()

    indices = random.sample(range(len(dataset)), k=n_images)

    fig, axes = plt.subplots(2,4,figsize=(13,7))

    axes = axes.ravel()

    with torch.no_grad():

        for ax, idx in zip(axes, indices):

            image, label = dataset[idx]

            outputs = model(image.unsqueeze(0).to(device))

            pred = outputs.argmax(dim=1).item()

            image = denormalize_image(image)

            ax.imshow(image.permute(1,2,0))

            ax.axis("off")

            color = "green" if pred==label else "red"

            ax.set_title(
                f"Pred:{class_names[pred]}\nTrue:{class_names[label]}",
                color=color,
                fontsize=9,
            )

    plt.tight_layout()

    plt.savefig(
        PREDICTIONS_PLOT_PATH,
        dpi=160,
    )

    plt.show()

In [ ]:
show_random_predictions(
    model,
    test_dataset,
    class_names,
    DEVICE,
)